# Deep Evidential Regression — Bike Sharing (LSTM)

In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, json, os, sys
from sklearn.metrics import r2_score
sys.path.insert(0, os.path.join('.', '..'))
from shared.data_utils import load_bike_sharing

CONFIG = {'method': 'deep_evidential', 'lstm_hidden': 64, 'lstm_layers': 1,
          'epochs': 100, 'batch_size': 64, 'lr': 1e-3, 'lambda_reg': 0.01, 'seeds': [42, 43, 44]}
RESULT_DIR = os.path.join('.', 'results', 'deep_evidential')
os.makedirs(RESULT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
class EvidentialLSTM(nn.Module):
    def __init__(self, n_features, lstm_hidden, lstm_layers):
        super().__init__()
        self.lstm = nn.LSTM(n_features, lstm_hidden, lstm_layers, batch_first=True)
        self.head = nn.Linear(lstm_hidden, 4)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.head(lstm_out[:, -1, :])
        gamma, nu, alpha, beta = out[:, 0:1], out[:, 1:2], out[:, 2:3], out[:, 3:4]
        return gamma, F.softplus(nu) + 1, F.softplus(alpha) + 1, F.softplus(beta)

def evidential_loss(y, gamma, nu, alpha, beta, lam):
    v = 2 * beta * (1 + nu) / (nu * alpha)
    nll = 0.5 * torch.log(torch.pi / nu) - alpha * torch.log(v) + (alpha + 0.5) * torch.log(
        nu * (y - gamma) ** 2 + v) + torch.lgamma(alpha) - torch.lgamma(alpha + 0.5)
    reg = lam * (torch.abs(y - gamma) * (2 * nu + alpha)).mean()
    return nll.mean() + reg, nu.squeeze(-1)
print('Model defined.')

In [ ]:
def train_one_seed(seed):
    print(f'\n--- Seed {seed} ---')
    X_train, y_train, X_val, y_val, X_test, y_test, scaler, (seq_len, n_feat) = \
        load_bike_sharing(random_state=seed)
    train_ds = torch.utils.data.TensorDataset(X_train, y_train)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
    model = EvidentialLSTM(n_feat, CONFIG['lstm_hidden'], CONFIG['lstm_layers']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    best_state, best_val = None, float('inf')
    for _ in range(CONFIG['epochs']):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            g, n, a, b = model(xb)
            loss, _ = evidential_loss(yb, g, n, a, b, CONFIG['lambda_reg'])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            g, n, a, b = model(X_val.to(DEVICE))
            vl, _ = evidential_loss(y_val.to(DEVICE), g, n, a, b, CONFIG['lambda_reg'])
        if vl.item() < best_val: best_val = vl.item(); best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        gamma, nu, _, _ = model(X_test.to(DEVICE))
    return gamma.cpu().numpy().squeeze(), nu.cpu().numpy().squeeze(), y_test.numpy().squeeze()

for seed in CONFIG['seeds']:
    y_pred, scores, y_true = train_one_seed(seed)
    sd = os.path.join(RESULT_DIR, f'seed_{seed}'); os.makedirs(sd, exist_ok=True)
    np.save(os.path.join(sd, 'test_predictions.npy'), y_pred)
    np.save(os.path.join(sd, 'test_scores.npy'), scores)
    np.save(os.path.join(sd, 'test_labels.npy'), y_true)
    print(f'  R²: {r2_score(y_true, y_pred):.4f}')
print(f'\nDone. Saved to {RESULT_DIR}')